# CS2 Tournament Simulator: Setup and Data Validation

This notebook validates the project environment, loads the initial feature table, and applies an explicit column allowlist before modeling begins.

# סימולטור טורנירי CS2: הקמה ובדיקת תקינות הנתונים

מחברת זו מאמתת את סביבת הפרויקט, טוענת את טבלת הפיצ'רים בגרסה הראשונה ומחילה רשימת עמודות מותרות לפני שלב המידול.

## Environment Verification

All core libraries are imported and their versions are recorded for reproducibility. An import failure stops the workflow before partial or misleading outputs can be created.

## אימות סביבת העבודה

לפני קריאת הנתונים אנו מייבאים את כל ספריות הליבה ומדפיסים את גרסאותיהן. רישום הגרסאות הוא חלק מיכולת השחזור: שינוי משמעותי ב־pandas, ב־scikit-learn או ב־XGBoost עלול להשפיע על טעינת הארטיפקט, על טיפול בערכים חסרים או על תוצאות האימון. כשל import בשלב זה עוצר את התהליך לפני שנוצרות תוצאות חלקיות.

In [1]:
import platform
from importlib.metadata import version
from pathlib import Path

import jupyter
import matplotlib
import numpy as np
import optuna
import pandas as pd
import rapidfuzz
import seaborn as sns
import sklearn
import xgboost

package_versions = {
    "Python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scikit-learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
    "jupyter": version("jupyter"),
    "matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
    "rapidfuzz": rapidfuzz.__version__,
    "optuna": optuna.__version__,
}

for package_name, package_version in package_versions.items():
    print(f"{package_name}: {package_version}")

Python: 3.12.13
pandas: 3.0.5
numpy: 2.5.3
scikit-learn: 1.9.1
xgboost: 3.4.1
jupyter: 1.1.1
matplotlib: 3.11.2
seaborn: 0.13.2
rapidfuzz: 3.14.6
optuna: 5.0.0


## Loading the Source Table

The path works from either the project root or `notebooks/`. The full table is loaded first so required columns can be validated explicitly before filtering.

## טעינת טבלת המקור

הנתיב נפתר הן כאשר המחברת מורצת משורש הפרויקט והן מתוך `notebooks/`. בשלב זה נטענת הטבלה המלאה ללא בחירת פיצ'רים, כדי שנוכל לאמת במפורש שהעמודות הדרושות קיימות לפני הסינון. המספר המודפס הוא בדיקת sanity ראשונית ולא מטריצת המודל הסופית.

In [2]:
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

feature_path = project_root / "data" / "final_tournament_features.csv"
if not feature_path.is_file():
    raise FileNotFoundError(f"Feature table not found: {feature_path}")

raw_df = pd.read_csv(feature_path, low_memory=False)
print(f"Loaded {feature_path.name}: {raw_df.shape}")

Loaded final_tournament_features.csv: (11151, 112)


## Feature Allowlist and Leakage Prevention

Only identifiers, time, map, target, and the original DNA candidates are retained. Post-match statistics are excluded; later ablations ultimately restrict production to six canonical Elo features.

## רשימת פיצ'רים מותרת ומניעת דליפה

ה־allowlist שומר רק מזהים, זמן, שם מפה, תווית היעד ופיצ'רי DNA שהיו מיועדים למחקר הראשוני. סטטיסטיקות שנוצרות לאחר המשחק—כגון תוצאה, kills, deaths, assists, ADR, KAST ו־K/D difference—אינן יכולות להיכנס במקרה למודל. בהמשך הפרויקט מחקרי ההסרה הראו שגם DNA אינו משפר את Elo הקנוני, ולכן המודל הייצורי משתמש לבסוף בששת פיצ'רי ה־Elo בלבד; התא הזה נשמר כתיעוד של שער הבטיחות הראשון.

In [3]:
identifier_columns = [
    "match_id",
    "game_id",
    "tournament",
    "team1_id",
    "team1",
    "team2_id",
    "team2",
    "map_id",
    "team1_join_key",
    "team2_join_key",
]
context_columns = ["datetime", "map_name", "team1_win"]
team1_dna_columns = [
    "team1_pistol_round_win_rate",
    "team1_n_pistol_rounds",
    "team1_ct_win_rate",
    "team1_n_ct_rounds",
    "team1_t_win_rate",
    "team1_n_t_rounds",
]
team2_dna_columns = [column.replace("team1_", "team2_") for column in team1_dna_columns]
dna_columns = team1_dna_columns + team2_dna_columns
source_allowlist = identifier_columns + context_columns + dna_columns

missing_columns = sorted(set(source_allowlist) - set(raw_df.columns))
if missing_columns:
    raise ValueError(f"Required allowlist columns are missing: {missing_columns}")

allowed_df = raw_df.loc[:, source_allowlist].copy()
allowed_df["datetime"] = pd.to_datetime(allowed_df["datetime"], errors="raise")
allowed_df["team1_has_dna"] = allowed_df[team1_dna_columns].notna().all(axis=1)
allowed_df["team2_has_dna"] = allowed_df[team2_dna_columns].notna().all(axis=1)

post_match_tokens = (
    "score", "kills", "deaths", "assists", "adr", "kast",
    "kddiff", "games_played", "player1", "player2", "player3",
    "player4", "player5",
)
leaked_columns = [
    column for column in allowed_df.columns
    if any(token in column.lower() for token in post_match_tokens)
]
if leaked_columns:
    raise AssertionError(f"Post-match columns passed the allowlist: {leaked_columns}")

if not allowed_df["team1_win"].dropna().isin([0, 1]).all():
    raise ValueError("team1_win contains values outside {0, 1}.")

print(f"Allowed dataframe shape: {allowed_df.shape}")
print(f"Team 1 DNA available: {allowed_df['team1_has_dna'].mean():.1%}")
print(f"Team 2 DNA available: {allowed_df['team2_has_dna'].mean():.1%}")
print("\nAllowed dataframe info:")
allowed_df.info()

Allowed dataframe shape: (11151, 27)
Team 1 DNA available: 70.5%
Team 2 DNA available: 69.4%

Allowed dataframe info:
<class 'pandas.DataFrame'>
RangeIndex: 11151 entries, 0 to 11150
Data columns (total 27 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   match_id                     11151 non-null  int64         
 1   game_id                      9969 non-null   float64       
 2   tournament                   11151 non-null  str           
 3   team1_id                     11151 non-null  int64         
 4   team1                        11151 non-null  str           
 5   team2_id                     11151 non-null  int64         
 6   team2                        11151 non-null  str           
 7   map_id                       8395 non-null   float64       
 8   team1_join_key               11151 non-null  str           
 9   team2_join_key               11151 non-null  str           
 10 